# **Question 1**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import load_iris

def load_data():
    data=load_iris()
    x=data.data
    y=data.target.reshape(-1, 1)
    enc=OneHotEncoder(sparse_output=False)
    y=enc.fit_transform(y)
    scl=StandardScaler()
    x=scl.fit_transform(x)
    return train_test_split(x,y,test_size=0.2,random_state=42)

def init_weights(inn, hid, out):
    np.random.seed(42)
    w1=np.random.randn(inn, hid)
    b1=np.zeros((1, hid))
    w2=np.random.randn(hid, out)
    b2=np.zeros((1, out))
    return w1,b1,w2,b2

def sig(x):
    return 1/(1+np.exp(-x))

def sigder(x):
    return x*(1-x)

def smx(x):
    expx=np.exp(x-np.max(x,axis=1,keepdims=True))
    return expx/np.sum(expx,axis=1,keepdims=True)

def train(xtr, ytr, w1, b1, w2, b2, lr, ep):
    for i in range(ep):
        z1=np.dot(xtr, w1) + b1
        a1=sig(z1)
        z2=np.dot(a1, w2) + b2
        a2=smx(z2)
        loss=-np.mean(ytr * np.log(a2 + 1e-9))
        dz2=a2 - ytr
        dw2=np.dot(a1.T, dz2) / xtr.shape[0]
        db2=np.sum(dz2, axis=0, keepdims=True) / xtr.shape[0]
        da1=np.dot(dz2, w2.T)
        dz1=da1 * sigder(a1)
        dw1=np.dot(xtr.T, dz1) / xtr.shape[0]
        db1=np.sum(dz1, axis=0, keepdims=True) / xtr.shape[0]
        w1-=lr * dw1
        b1-=lr * db1
        w2-=lr * dw2
        b2-=lr * db2
        if i % 100 == 0:
          print(f"Epoch {i}, Loss: {loss:.4f}")
    return w1, b1, w2, b2

def evaluate(xte, yte, w1, b1, w2, b2, str1):
    z1=np.dot(xte, w1) + b1
    a1=sig(z1)
    z2=np.dot(a1, w2) + b2
    a2=smx(z2)
    yp=np.argmax(a2, axis=1)
    yt=np.argmax(yte, axis=1)
    acc=np.mean(yp==yt)
    print(f"{str1} {acc:.4f}")

xtr,xte,ytr,yte=load_data()
inn,hid,out=xtr.shape[1], 10, ytr.shape[1]
w1,b1,w2,b2=init_weights(inn, hid, out)
w1,b1,w2,b2=train(xtr, ytr, w1, b1, w2, b2, lr=0.1, ep=1000)
evaluate(xte, yte, w1, b1, w2, b2, "Testing Accuracy:")

Epoch 0, Loss: 0.4063
Epoch 100, Loss: 0.1321
Epoch 200, Loss: 0.1059
Epoch 300, Loss: 0.0907
Epoch 400, Loss: 0.0790
Epoch 500, Loss: 0.0694
Epoch 600, Loss: 0.0614
Epoch 700, Loss: 0.0548
Epoch 800, Loss: 0.0495
Epoch 900, Loss: 0.0452
Testing Accuracy: 1.0000


# **Question 2**

In [ ]:
# @title Creating Dataset
data = {
    "World": [
        "World leaders sign a new climate agreement",
        "World leaders sign a new peace treaty",
        "World leaders sign a new trade deal",
        "World leaders discuss nuclear disarmament",
        "World leaders discuss economic policies",
        "World leaders discuss global security concerns",
        "A global crisis leads to humanitarian aid",
        "A global crisis leads to refugee displacement",
        "A global crisis leads to economic downturn",
        "A global summit focuses on human rights",
        "A global summit focuses on climate change",
        "A global summit focuses on poverty reduction",
        "International relations improve after diplomatic talks",
        "International relations worsen due to border tensions",
        "International relations strengthen through new alliances",
        "Natural disasters impact multiple regions",
        "Natural disasters increase due to climate change",
        "Natural disasters lead to economic losses",
        "Countries pledge to reduce carbon emissions",
        "Countries pledge to increase renewable energy use",
        "Countries pledge to improve diplomatic ties",
        "A political leader resigns amid corruption allegations",
        "A political leader is elected after a historic vote",
        "A political leader announces new foreign policies"
    ],
    "Sports": [
        "A football team wins the championship",
        "A football team wins an international tournament",
        "A football team wins a local league title",
        "A tennis player sets a new record",
        "A tennis player wins a major tournament",
        "A tennis player reaches the finals",
        "A basketball team wins a historic game",
        "A basketball team qualifies for the playoffs",
        "A basketball team secures a dramatic victory",
        "A legendary athlete announces retirement",
        "A legendary athlete makes a comeback",
        "A legendary athlete breaks a record",
        "An underdog team wins against all odds",
        "An underdog team reaches the finals",
        "An underdog team stuns the competition",
        "A new sports league is launched",
        "A new sports league attracts global attention",
        "A new sports league introduces innovative rules",
        "An esports team dominates an international event",
        "An esports team wins a world championship",
        "An esports team secures a major sponsorship",
        "A gymnast performs a record-breaking routine",
        "A gymnast wins an Olympic gold medal",
        "A gymnast stuns the audience with a flawless routine"
    ],
    "Business": [
        "Stock markets hit a new high",
        "Stock markets hit a new low",
        "Stock markets experience heavy volatility",
        "A tech company launches a new product",
        "A tech company announces a major merger",
        "A tech company expands to international markets",
        "A major corporation announces layoffs",
        "A major corporation faces financial struggles",
        "A major corporation reports record profits",
        "Investors focus on renewable energy projects",
        "Investors focus on cryptocurrency markets",
        "Investors focus on real estate growth",
        "Cryptocurrency prices surge unexpectedly",
        "Cryptocurrency prices crash amid regulations",
        "Cryptocurrency markets face uncertainty",
        "A startup gains major investments",
        "A startup disrupts the traditional industry",
        "A startup launches an innovative product",
        "E-commerce platforms report record sales",
        "E-commerce platforms expand their services",
        "E-commerce platforms face regulatory scrutiny",
        "A billionaire entrepreneur announces a new venture",
        "A billionaire entrepreneur donates to charity",
        "A billionaire entrepreneur invests in AI technology"
    ],
    "Science/Technology": [
        "AI is transforming the healthcare industry",
        "AI is transforming the finance industry",
        "AI is transforming the education industry",
        "NASA discovers a new exoplanet",
        "NASA discovers a new galaxy",
        "NASA discovers a new asteroid",
        "A breakthrough in quantum computing",
        "A breakthrough in fusion energy",
        "A breakthrough in medical research",
        "Electric vehicles gain popularity",
        "Electric vehicles become more affordable",
        "Electric vehicles improve battery technology",
        "Scientists develop a revolutionary vaccine",
        "Scientists develop a new cancer treatment",
        "Scientists develop a new gene-editing technique",
        "A cybersecurity breach exposes sensitive data",
        "A cybersecurity breach affects millions of users",
        "A cybersecurity breach raises privacy concerns",
        "A self-driving car completes a test drive",
        "A self-driving car improves navigation systems",
        "A self-driving car faces regulatory challenges",
        "SpaceX launches a new rocket",
        "SpaceX successfully lands a reusable rocket",
        "SpaceX announces plans for Mars colonization"
    ]
}


In [ ]:
# @title Transforming Dataset into Bag of Words
import numpy as np
import re
from collections import Counter

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()

vocab=set()
for c in data:
    for s in data[c]:
        vocab.update(preprocess(s))
vocab=list(vocab)

def wordBag(text, vocab):
    words=preprocess(text)
    bow=np.zeros(len(vocab))
    c=Counter(words)
    for i, word in enumerate(vocab):
        bow[i]=c[word]
    return bow

X=[]
Y=[]
categories=list(data.keys())
for c,s in data.items():
    for s1 in s:
        X.append(wordBag(s1,vocab))
        Y.append(categories.index(c))

In [ ]:
# @title Transforming X,Y into Arrays
X=np.array(X)
Y=np.array(Y)
Y=Y.reshape(-1,1)
enc=OneHotEncoder(sparse_output=False)
Y=enc.fit_transform(Y)

In [ ]:
# @title Neural Network
import numpy as np
def init_weights(inn, hid1, hid2, out):
    np.random.seed(42)
    w1 = np.random.randn(inn, hid1)
    b1 = np.zeros((1, hid1))
    w2 = np.random.randn(hid1, hid2)
    b2 = np.zeros((1, hid2))
    w3 = np.random.randn(hid2, out)
    b3 = np.zeros((1, out))
    return w1, b1, w2, b2, w3, b3
def sig(x):
    return 1 / (1 + np.exp(-x))
def sigder(x):
    return x * (1 - x)
def smx(x):
    expx = np.exp(x - np.max(x, axis=1, keepdims=True))
    return expx / np.sum(expx, axis=1, keepdims=True)
def train(xtr, ytr, w1, b1, w2, b2, w3, b3, lr, ep):
    for i in range(ep):
        z1 = np.dot(xtr, w1) + b1
        a1 = sig(z1)
        z2 = np.dot(a1, w2) + b2
        a2 = sig(z2)
        z3 = np.dot(a2, w3) + b3
        a3 = smx(z3)
        loss = -np.mean(ytr * np.log(a3 + 1e-9))
        dz3 = a3 - ytr
        dw3 = np.dot(a2.T, dz3) / xtr.shape[0]
        db3 = np.sum(dz3, axis=0, keepdims=True) / xtr.shape[0]
        da2 = np.dot(dz3, w3.T)
        dz2 = da2 * sigder(a2)
        dw2 = np.dot(a1.T, dz2) / xtr.shape[0]
        db2 = np.sum(dz2, axis=0, keepdims=True) / xtr.shape[0]
        da1 = np.dot(dz2, w2.T)
        dz1 = da1 * sigder(a1)
        dw1 = np.dot(xtr.T, dz1) / xtr.shape[0]
        db1 = np.sum(dz1, axis=0, keepdims=True) / xtr.shape[0]
        w1 -= lr * dw1
        b1 -= lr * db1
        w2 -= lr * dw2
        b2 -= lr * db2
        w3 -= lr * dw3
        b3 -= lr * db3
        if i % 1000 == 0:
            print(f"Epoch {i}, Loss: {loss:.4f}")
    return w1, b1, w2, b2, w3, b3

def predict(x, w1, b1, w2, b2, w3, b3):
    z1 = np.dot(x, w1) + b1
    a1 = sig(z1)
    z2 = np.dot(a1, w2) + b2
    a2 = sig(z2)
    z3 = np.dot(a2, w3) + b3
    a3 = smx(z3)
    return np.argmax(a3, axis=1)

def evaluate(x, y, w1, b1, w2, b2, w3, b3):
    yp = predict(x, w1, b1, w2, b2, w3, b3)
    y_labels = np.argmax(y, axis=1)
    print("Actual Lables:",y_labels)
    print("Predicted Labels:",yp)
    acc = np.mean(yp == y_labels)
    print(f"Test Accuracy: {acc:.4f}")


In [ ]:
# @title Prediction and Results
from sklearn.model_selection import train_test_split
xtr, xte, ytr, yte = train_test_split(X, Y, test_size=0.1, random_state=42)
inp, hid1, hid2, out = xtr.shape[1], 96, 32, 4
w1, b1, w2, b2, w3, b3 = init_weights(inp, hid1, hid2, out)
w1, b1, w2, b2, w3, b3 = train(xtr, ytr, w1, b1, w2, b2, w3, b3, lr=0.5, ep=10000)
evaluate(xte, yte, w1, b1, w2, b2, w3, b3)

Epoch 0, Loss: 1.4079
Epoch 1000, Loss: 0.0013
Epoch 2000, Loss: 0.0006
Epoch 3000, Loss: 0.0004
Epoch 4000, Loss: 0.0003
Epoch 5000, Loss: 0.0002
Epoch 6000, Loss: 0.0002
Epoch 7000, Loss: 0.0001
Epoch 8000, Loss: 0.0001
Epoch 9000, Loss: 0.0001
Actual Lables: [3 3 3 3 1 3 2 1 0 0]
Predicted Labels: [3 3 3 3 1 3 2 1 0 0]
Test Accuracy: 1.0000
